# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 3): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")

# Dates and locations
locations = "Antarctica,North America,South America"
start_date = "11-01-2021"
end_date = "01-02-2026"
date_range = start_date + "--" + end_date


# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

genotypes_df = pd.read_excel("genotype_key.xlsx")
genotypes = list(genotypes_df["Genotype"])

# serotype = "H5N1"
# genotypes = ["B3.13", "D1.1", "D1.3"]
# genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]


## Downloading Data

In [3]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Get files
            open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 

## De-Duplication

In [4]:
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments

122194


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
18,PQ284576.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Sterna hirundo,NaN,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8
19,PQ284577.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Sterna hirundo,NaN,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8
20,PQ284578.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Sterna hirundo,NaN,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8
21,PQ284579.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Sterna hirundo,NaN,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8
22,PQ284580.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Sterna hirundo,NaN,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114200,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
114201,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
114202,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8
114203,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Gallus gallus,allantoic fluid,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8


## Add sequences to dataframe

In [5]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title | Accession

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
# sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
# sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-1][:-1])

print(sequences_fasta["full_header"])

# Extract segment number so that we can add the correct sequences to the correct sample
# sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["Accession"]) # , "Segment"])

0         >Influenza A virus |Chile||H5N1|2025-01-13|Ste...
1         >Influenza A virus |Chile||H5N1|2025-01-13|Ste...
2         >Influenza A virus |Chile||H5N1|2025-01-13|Ste...
3         >Influenza A virus |Chile||H5N1|2025-01-13|Ste...
4         >Influenza A virus |Chile||H5N1|2025-01-13|Ste...
                                ...                        
122189    >Influenza A virus |Mexico: Veracruz|28159-398...
122190    >Influenza A virus |Mexico: Veracruz|28159-398...
122191    >Influenza A virus |Mexico: Veracruz|28159-398...
122192    >Influenza A virus |Mexico: Veracruz|28159-398...
122193    >Influenza A virus |Mexico: Veracruz|28159-398...
Name: full_header, Length: 122194, dtype: object
122194
                                         full_header  \
0  >Influenza A virus |Chile||H5N1|2025-01-13|Ste...   
1  >Influenza A virus |Chile||H5N1|2025-01-13|Ste...   
2  >Influenza A virus |Chile||H5N1|2025-01-13|Ste...   
3  >Influenza A virus |Chile||H5N1|2025-01-13|Ste...   
4  >

In [6]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PQ284576.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAAC...
1,PQ284577.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...
2,PQ284578.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTACTGATTCAAAATGGAAGACTTTGTGCGACAAT...
3,PQ284579.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGGGTTCACTCTGTCAAAATGGAGAACATAGTACTA...
4,PQ284580.1,GenBank,GCA_054311525.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Oliveira,C.H.S....","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,NaN,2024-04-05,2025-12-31,ssRNA(-),8,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCGAAAGCAGGGTAGATAATCACTCACTGAGTGACATTCACATCA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105523,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
105524,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
105525,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
105526,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


## Find genotype using old genoflu results or Andersen Lab genoflu output

Old genoflu results should accumulate into one file to avoid having to genotype anything again.

In [7]:
genoflu_old = pd.read_csv("output.tsv", delimiter="\t")
genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"}) # So we can merge
# print(genoflu_old)
metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
metadata_segments["Partial_Header"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", "")) # Get rid of header indicator
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]: # Forbidden punctuation
    metadata_segments["Partial_Header"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(c, "_"))
# metadata_segments["Accession_Root"] = metadata_segments["Accession"].values[0].split(".")[0]
# print(metadata_segments["Partial_Header"])

# Merge to get already-genotyped segments
metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header") # .dropna(subset="SRA_Accession")
# Isolate not-already-genotyped segments
metadata_segments_new = metadata_segments.merge(genoflu_old,indicator = True, how='left', on="Partial_Header").loc[lambda x : x['_merge']!='both'] # .dropna(subset="SRA_Accession")

print(len(metadata_segments_old))
print(len(metadata_segments_new))

101088
4640


In [26]:
metadata_segments_new[metadata_segments_new["Isolate"] == "A273"]

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Partial_Header,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Date run,_merge,Strain
104123,OQ747762.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104127,OQ747769.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104131,OQ747882.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104143,OQ747867.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104147,OQ747877.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...
104151,OQ747893.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104163,OQ747888.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
104167,OQ747899.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...


In [9]:
# Get genoflu results from Andersen and see if any fit

os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"}) # So we can merge

# Find those genotyped by Andersen via merge
metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
# Keep all known genotypes
metadata_segments_known = pd.merge(metadata_segments_old, metadata_segments_known_andersen, how="left") # .drop_duplicates(keep="last", inplace=True) # Since we know both of these

print(genoflu_andersen)

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PX748036.1        GenBank  GCA_054196825.1   SRR36162763  SAMN53332038   
1     PX748037.1        GenBank  GCA_054196825.1   SRR36162763  SAMN53332038   
2     PX748038.1        GenBank  GCA_054196825.1   SRR36162763  SAMN53332038   
3     PX748039.1        GenBank  GCA_054196825.1   SRR36162763  SAMN53332038   
4     PX748040.1        GenBank  GCA_054196825.1   SRR36162763  SAMN53332038   
...          ...            ...              ...           ...           ...   
4435  PQ135389.1        GenBank  GCA_041199025.1   SRR29694510  SAMN42278998   
4436  PQ135390.1        GenBank  GCA_041199025.1   SRR29694510  SAMN42278998   
4437  PQ135391.1        GenBank  GCA_041199025.1   SRR29694510  SAMN42278998   
4438  PQ135392.1        GenBank  GCA_041199025.1   SRR29694510  SAMN42278998   
4439  PQ135393.1        GenBank  GCA_041199025.1   SRR29694510  SAMN42278998   

        BioProject      Organism_Name  

In [27]:
metadata_segments_unknown = metadata_segments_new # Variable rename so we don't break old code

print(metadata_segments_unknown[metadata_segments_unknown["Isolate"] == "A273"])

         Accession GenBank_RefSeq Assembly SRA_Accession BioSample BioProject  \
104123  OQ747762.1        GenBank      NaN           NaN       NaN        NaN   
104127  OQ747769.1        GenBank      NaN           NaN       NaN        NaN   
104131  OQ747882.1        GenBank      NaN           NaN       NaN        NaN   
104143  OQ747867.1        GenBank      NaN           NaN       NaN        NaN   
104147  OQ747877.1        GenBank      NaN           NaN       NaN        NaN   
104151  OQ747893.1        GenBank      NaN           NaN       NaN        NaN   
104163  OQ747888.1        GenBank      NaN           NaN       NaN        NaN   
104167  OQ747899.1        GenBank      NaN           NaN       NaN        NaN   

            Organism_Name                         Species  \
104123  Influenza A virus  Alphainfluenzavirus influenzae   
104127  Influenza A virus  Alphainfluenzavirus influenzae   
104131  Influenza A virus  Alphainfluenzavirus influenzae   
104143  Influenza A virus 

## Create FASTA files of unknown genotypes 

In [11]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_unknown["Partial_Header"])

print(df_list[0]["full_header"])

# Make fasta files
for df in df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

0         >Influenza A virus |Brazil: SaoFranciscoItabap...
1         >Influenza A virus |Brazil: SaoFranciscoItabap...
2         >Influenza A virus |Brazil: SaoFranciscoItabap...
3         >Influenza A virus |Brazil: SaoFranciscoItabap...
4         >Influenza A virus |Brazil: SaoFranciscoItabap...
                                ...                        
104143    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
104147    >Influenza A virus |Peru|A273|H5N1|2022-12||Ge...
104151    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
104163    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
104167    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
Name: Partial_Header, Length: 4640, dtype: object
1192    >Influenza_A_virus__USA__IN_25G02195_004_origi...
1193    >Influenza_A_virus__USA__IN_25G02195_004_origi...
1194    >Influenza_A_virus__USA__IN_25G02195_004_origi...
1195    >Influenza_A_virus__USA__IN_25G02195_004_origi...
1196    >Influenza_A_virus__USA__IN_25G02195_004_origi...


## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "results.tsv" is in the downloads directory.



In [12]:
'''
To run GenoFLU-multi, first change directories to Multi-GenoFLU directory (and activate genoflu conda environment):

conda activate genoflu
cd GenoFLU-multi

And then call the python script:

python bin/genoflu-multi.py -f <FASTA_directory>
'''

'\nTo run GenoFLU-multi, first change directories to Multi-GenoFLU directory (and activate genoflu conda environment):\n\nconda activate genoflu\ncd GenoFLU-multi\n\nAnd then call the python script:\n\npython bin/genoflu-multi.py -f <FASTA_directory>\n'

In [32]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t")

# Create partial headers to merge genoflu results with previously unknown segments
metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
    metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(c, "_"))

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(">", ""))

metadata_segments_unknown["Strain"] = metadata_segments_unknown["Partial_Header"] # Renaming so we can merge

# Merge
metadata_genoflu = metadata_segments_unknown.merge(output_genoflu, how="left", on="Strain", suffixes=('_left', '_right')) 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")

print(metadata_genoflu[metadata_genoflu["Isolate"] == "A273"])

print(metadata_genoflu["Genotype_x"])

print(metadata_genoflu.columns)


       Accession GenBank_RefSeq Assembly SRA_Accession BioSample BioProject  \
4632  OQ747762.1        GenBank      NaN           NaN       NaN        NaN   
4633  OQ747769.1        GenBank      NaN           NaN       NaN        NaN   
4634  OQ747882.1        GenBank      NaN           NaN       NaN        NaN   
4635  OQ747867.1        GenBank      NaN           NaN       NaN        NaN   
4636  OQ747877.1        GenBank      NaN           NaN       NaN        NaN   
4637  OQ747893.1        GenBank      NaN           NaN       NaN        NaN   
4638  OQ747888.1        GenBank      NaN           NaN       NaN        NaN   
4639  OQ747899.1        GenBank      NaN           NaN       NaN        NaN   

          Organism_Name                         Species                Genus  \
4632  Influenza A virus  Alphainfluenzavirus influenzae  Alphainfluenzavirus   
4633  Influenza A virus  Alphainfluenzavirus influenzae  Alphainfluenzavirus   
4634  Influenza A virus  Alphainfluenzavirus inf

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_24504\3633387093.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill(limit_area="inside")


# Concatenate with known genotypes

In [33]:
# Rename columns so we can concatenate
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

# Concatenation
metadata_genoflu = pd.concat([metadata_genoflu, metadata_segments_known])

metadata_genoflu.columns

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Partial_Header', 'Genotype_y',
       'Genotype List Used, >=98.0%_left', 'Genotype Sample Title List_left',
       'Genotype Percent Match List_left', 'Genotype Mismatch List_left',
       'Genotype Average Depth of Coverage List_left', 'Date run_left',
       '_merge', 'Strain', 'Genotype', 'Genotype List Used, >=98.0%_right',
       'Genotype Sample Title List_right', 'Genotype Percent Match List_right',
       'Genotype Mismatch List_right',
       'Genotype Average Depth of Coverage List_right', 'Date run_

In [34]:
metadata_segments_known # .dropna(subset="SRA_Accession")

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Genotype_official,Serotype
0,PV659823.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
1,PV659824.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
2,PV659825.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
3,PV659826.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
4,PV659827.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101083,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101084,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101085,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101086,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2


In [35]:
# Cut down to only columns we want
metadata_genoflu = metadata_genoflu[["Accession", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header", "Strain"]]

# Get genbank strain name
metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] )
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1])

metadata_genoflu # [metadata_genoflu["Genotype"]  == "B3.13"]

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name
0,PQ284576.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAAC...,H5N1,1,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...
1,PQ284577.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...
2,PQ284578.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTACTGATTCAAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...
3,PQ284579.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGGGTTCACTCTGTCAAAATGGAGAACATAGTACTA...,H5N1,4,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...
4,PQ284580.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCGAAAGCAGGGTAGATAATCACTCACTGAGTGACATTCACATCA...,H5N1,5,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101083,OK205883.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,H5N2,4,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101084,OK205884.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,H5N2,5,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101085,OK205885.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,H5N2,6,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101086,OK205886.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,H5N2,7,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995


In [36]:
metadata_genoflu["Serotype"]

0         H5N1
1         H5N1
2         H5N1
3         H5N1
4         H5N1
          ... 
101083    H5N2
101084    H5N2
101085    H5N2
101086    H5N2
101087    H5N2
Name: Serotype, Length: 105728, dtype: object

## Relabeling Sequences

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [37]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['domestic cat', 'chukar', 'ring-necked pheasant', 'seabird', 'snowy owl', 'wild bird', 'black bear', 'peacock', 'emu', 'ferruginous hawk', 'gray seal', 'dog', 'pelecanus thagus', 'mountain lion', 'red-necked phalarope', 'nasua nasua', 'baikal teal', 'sharp-shinned hawk', 'blue-winged and cinnamon teal', 'lesser scaup', 'black scoter', 'crested caracara', 'lynx', 'goat', 'fregata magnificens', 'lesser snow goose blue-morph', 'american crow', 'raptor', 'alpaca', 'american coot', 'guinea fowl', 'waterfowl', 'turkey vulture', 'eared grebe', 'crane', 'gallus', 'pintail', 'american woodcock', 'brant', 'swine', 'common goldeneye', 'sandhill crane', 'ruffed grouse', 'green heron', 'polar bear', 'common barn owl', 'serval', 'sandwich tern', 'redhead duck', 'south american sea lion', 'domestic duck', 'black swift', 'greater white-fronted goose', 'broad-winged hawk', 'great-horned owl', 'american blue-winged teal', 'blue-winged teal', 'willet', 'pigeon', 'western sandpiper', 'herring gull', 'osp

In [38]:
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Unassigned" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [39]:
metadata_genoflu

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name,Genotype
0,PQ284576.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATAAAAGAAC...,H5N1,1,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...,B3.2
1,PQ284577.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...,B3.2
2,PQ284578.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGTACTGATTCAAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...,B3.2
3,PQ284579.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCAAAAGCAGGGGTTCACTCTGTCAAAATGGAGAACATAGTACTA...,H5N1,4,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...,B3.2
4,PQ284580.1,Influenza A virus (A/Sterna hirundo/SaoFrancis...,Sterna hirundo,2024-04-05,NaN,1060-N-2024,B3.2,Brazil: SaoFranciscoItabapoanaBR,>Influenza A virus |Brazil: SaoFranciscoItabap...,AGCGAAAGCAGGGTAGATAATCACTCACTGAGTGACATTCACATCA...,H5N1,5,NaN,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,Influenza_A_virus__Brazil__SaoFranciscoItabapo...,A/Sterna hirundo/SaoFranciscoItabapoanaBR/1060...,B3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101083,OK205883.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,H5N2,4,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
101084,OK205884.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,H5N2,5,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
101085,OK205885.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,H5N2,6,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned
101086,OK205886.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Unassigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,H5N2,7,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995,Unassigned


In [40]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Get geographic locations
metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        x
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


print(metadata_genoflu["Geo_Location_Abrv"])

0         Brazil-SaoFranciscoItabapoanaBR
1         Brazil-SaoFranciscoItabapoanaBR
2         Brazil-SaoFranciscoItabapoanaBR
3         Brazil-SaoFranciscoItabapoanaBR
4         Brazil-SaoFranciscoItabapoanaBR
                       ...               
101083                    Mexico-Veracruz
101084                    Mexico-Veracruz
101085                    Mexico-Veracruz
101086                    Mexico-Veracruz
101087                    Mexico-Veracruz
Name: Geo_Location_Abrv, Length: 105728, dtype: object


In [ ]:
metadata_genoflu['SRA_Accession'] = np.where(metadata_genoflu['SRA_Accession'] == "", metadata_genoflu['Accession'].apply(lambda x: x.split(".")[0]), metadata_genoflu['SRA_Accession'])

# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"].apply(lambda x: x.split(",")[0]) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names


print(metadata_genoflu)

         Accession                                      GenBank_Title  \
0       PQ284576.1  Influenza A virus (A/Sterna hirundo/SaoFrancis...   
1       PQ284577.1  Influenza A virus (A/Sterna hirundo/SaoFrancis...   
2       PQ284578.1  Influenza A virus (A/Sterna hirundo/SaoFrancis...   
3       PQ284579.1  Influenza A virus (A/Sterna hirundo/SaoFrancis...   
4       PQ284580.1  Influenza A virus (A/Sterna hirundo/SaoFrancis...   
...            ...                                                ...   
101083  OK205883.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
101084  OK205884.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
101085  OK205885.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
101086  OK205886.1  Influenza A virus (A/chicken/Veracruz/28159-39...   
101087  OK205887.1  Influenza A virus (A/chicken/Veracruz/28159-39...   

                  Host Collection_Date SRA_Accession      Isolate  \
0       sterna hirundo      2024-04-05      PQ284576  

In [42]:
# De-duplicate

print(len(metadata_genoflu))
metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["Isolate", "genbank_name", "Segment"], keep="last")
print(len(metadata_genoflu))

105728
105528


## Rename segments and make complete FASTA files

In [43]:
# Set up segments

genotypes.append("Unassigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

Minor07_PB2
Minor45_PB2
Minor14_PB2
B2.1_PB2
Minor28_PB2
Minor34_PB2
Unassigned_PB2
B3.12_PB2
Minor09_PB2
B3.3_PB2
Minor90_PB2
Minor50_PB2
D1.1_PB2
Minor63_PB2
B3.4_PB2
Minor91_PB2
B5.1_PB2
Minor11_PB2
Minor104_PB2
A2_PB2
B3.5_PB2
B2.2_PB2
Minor19_PB2
A3_PB2
Minor65_PB2
Minor13_PB2
Minor98_PB2
B1.1_PB2
Minor94_PB2
B3.10_PB2
C2.1_PB2
Minor81_PB2
B1.2_PB2
Minor77_PB2
Minor08_PB2
A1_PB2
Minor12_PB2
B3.1_PB2
Minor87_PB2
B3.2_PB2
A4_PB2
B3.7_PB2
Minor102_PB2
Minor04_PB2
B4.1_PB2
D1.3_PB2
B3.13_PB2
Minor101_PB2
A6_PB2
Minor01_PB2
A5_PB2
D1.2_PB2
B1.3_PB2
B3.6_PB2
Minor105_PB2
Minor07_PB1
Minor45_PB1
Minor14_PB1
B2.1_PB1
Minor28_PB1
Minor34_PB1
Unassigned_PB1
B3.12_PB1
Minor09_PB1
B3.3_PB1
Minor90_PB1
Minor50_PB1
D1.1_PB1
Minor63_PB1
B3.4_PB1
Minor91_PB1
B5.1_PB1
Minor11_PB1
Minor104_PB1
A2_PB1
B3.5_PB1
B2.2_PB1
Minor19_PB1
A3_PB1
Minor65_PB1
Minor13_PB1
Minor98_PB1
B1.1_PB1
Minor94_PB1
B3.10_PB1
C2.1_PB1
Minor81_PB1
B1.2_PB1
Minor77_PB1
Minor08_PB1
A1_PB1
Minor12_PB1
B3.1_PB1
Minor87_PB1
B3.

In [44]:
# Create FASTA files

os.chdir(complete_files)

for df in segment_genotype_dfs:
    print(df)
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

        Accession                                      GenBank_Title  \
15600  PV309253.1  Influenza A virus (A/Domestic bird/AB/FAV-0893...   
89864  OQ958737.1  Influenza A virus (A/Backyard bird/Nebraska/22...   

                Host Collection_Date SRA_Accession          Isolate  \
15600  domestic bird      2022-08-16      PV309253  FAV-0893-1-2022   
89864  backyard bird      2022-04-26      OQ958737    22-013105-001   

      Genotype_official   Geo_Location  \
15600           Minor07     Canada: AB   
89864           Minor07  USA: Nebraska   

                                             full_header  \
15600  >Influenza A virus |Canada: AB|FAV-0893-1-2022...   
89864  >Influenza A virus |USA: Nebraska|22-013105-00...   

                                                sequence  ... File Name  \
15600  ATGGAGAGAATAAAAGAACTAAGAGATCTAATGTCACAGTCTCGCA...  ...             
89864  ATGGAGAGAATAAAAGAACTAAGAGATCTAATGTCACAGTCTCGCA...  ...             

                                   